<div style="text-align: center; padding: 30px 0 10px 0;">

# Pontificia Universidad Católica de Chile
### Instituto de Ingeniería Matemática y Computacional
### Magíster en Inteligencia Artificial — MIA

---

# Tarea 1
## Fundamentos Matemáticos para la Inteligencia Artificial
### IMT3850 · 2025-1

---

| | |
|---|---|
| **Alumno** | Armando A.V. |
| **Profesor** | Manuel A. Sánchez |
| **Fecha** | Marzo 2026 |

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd, norm, matrix_rank

# Configuración global de gráficos
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)

---
## Problema 1 · Normas  
**(5 puntos)**


> **Enunciado.** Demuestre la siguiente identidad:
>
> $$\mathrm{rms}_w^2(\mathbf{x}) \;=\; \mathrm{avg}_w^2(\mathbf{x}) \;+\; \mathrm{std}_w^2(\mathbf{x})$$
>
> donde $w_1, w_2, \dots, w_n$ son pesos positivos con suma total $W = \sum_{i=1}^{n} w_i > 0$.


### Demostración

Partimos de las definiciones. Sean $w_1, \dots, w_n > 0$ con $W = \sum_{i=1}^n w_i$. Denotamos la media ponderada como $\bar{x}_w \,:=\, \mathrm{avg}_w(\mathbf{x})$.

$$
\mathrm{avg}_w(\mathbf{x}) = \frac{1}{W}\sum_{i=1}^{n} w_i\, x_i\,, \qquad
\mathrm{std}_w^2(\mathbf{x}) = \frac{1}{W}\sum_{i=1}^{n} w_i\,(x_i - \bar{x}_w)^2\,, \qquad
\mathrm{rms}_w^2(\mathbf{x}) = \frac{1}{W}\sum_{i=1}^{n} w_i\, x_i^2\,.
$$

Expandimos el cuadrado dentro de $\mathrm{std}_w^2$:

$$
\mathrm{std}_w^2(\mathbf{x})
= \frac{1}{W}\sum_{i=1}^{n} w_i \bigl(x_i^2 - 2\,x_i\,\bar{x}_w + \bar{x}_w^2\bigr)
= \underbrace{\frac{1}{W}\sum_i w_i\,x_i^2}_{\mathrm{rms}_w^2(\mathbf{x})}
  \;-\; 2\,\bar{x}_w\!\underbrace{\frac{1}{W}\sum_i w_i\,x_i}_{\bar{x}_w}
  \;+\; \bar{x}_w^2\!\underbrace{\frac{1}{W}\sum_i w_i}_{1}\,.
$$

Sustituyendo:

$$
\mathrm{std}_w^2(\mathbf{x}) = \mathrm{rms}_w^2(\mathbf{x}) - 2\,\bar{x}_w^2 + \bar{x}_w^2 = \mathrm{rms}_w^2(\mathbf{x}) - \bar{x}_w^2\,.
$$

Reordenando se obtiene la identidad deseada:

$$
\boxed{\;\mathrm{rms}_w^2(\mathbf{x}) = \mathrm{avg}_w^2(\mathbf{x}) + \mathrm{std}_w^2(\mathbf{x})\;}
\qquad \blacksquare
$$


### Verificación numérica


In [ ]:
def avg_w(x, w):
    """Media ponderada."""
    return np.sum(w * x) / np.sum(w)


def std2_w(x, w):
    """Varianza ponderada."""
    mu = avg_w(x, w)
    return np.sum(w * (x - mu) ** 2) / np.sum(w)


def rms2_w(x, w):
    """RMS² ponderado."""
    return np.sum(w * x ** 2) / np.sum(w)


# --- Datos de prueba ---
x = np.array([1.0, 3.0, 5.0, 7.0, 9.0])
w = np.array([0.5, 1.0, 1.5, 2.0, 0.3])

lhs = rms2_w(x, w)
rhs = avg_w(x, w) ** 2 + std2_w(x, w)

print(f"  rms²_w(x)              = {lhs:.12f}")
print(f"  avg²_w(x) + std²_w(x)  = {rhs:.12f}")
print(f"  |diferencia|           = {abs(lhs - rhs):.2e}")
print()
print(f"  ✓ Identidad verificada numéricamente." if np.isclose(lhs, rhs) else "  ✗ ERROR")


---
## Problema 2 · Aplicación del algoritmo K-means  
**(15 puntos)**


> **2 a)** Programe el algoritmo de K-means. Construya una rutina
> `k_means_fit(X, Z0, NITERMAX)` donde `X` son los datos, `Z0` los representantes iniciales
> y `NITERMAX` el número máximo de iteraciones.


### 2 a) Implementación de `k_means_fit`


In [ ]:
def k_means_fit(X, Z0, NITERMAX):
    """
    Algoritmo de K-means.

    Parámetros
    ----------
    X        : ndarray (N, d)  –  Datos de entrada.
    Z0       : ndarray (k, d)  –  Representantes iniciales.
    NITERMAX : int             –  Número máximo de iteraciones.

    Retorna
    -------
    Z      : ndarray (k, d)  –  Representantes finales.
    labels : ndarray (N,)    –  Etiqueta asignada a cada dato.
    J_hist : list[float]     –  Valor de J_clust en cada iteración.
    """
    N, d = X.shape
    k    = Z0.shape[0]
    Z    = Z0.copy()
    J_hist = []

    for it in range(NITERMAX):
        # 1. Asignación: cada dato al representante más cercano
        dists  = np.linalg.norm(X[:, None, :] - Z[None, :, :], axis=2)  # (N, k)
        labels = np.argmin(dists, axis=1)

        # Función objetivo
        J = np.sum(dists[np.arange(N), labels] ** 2)
        J_hist.append(J)

        # 2. Actualización: nuevo centroide = media del cluster
        Z_new = np.empty_like(Z)
        for j in range(k):
            mask = labels == j
            Z_new[j] = X[mask].mean(axis=0) if mask.any() else Z[j]

        # Criterio de parada
        if np.allclose(Z, Z_new):
            break
        Z = Z_new

    return Z, labels, J_hist


**Prueba rápida con datos sintéticos** para confirmar que el algoritmo opera correctamente.


In [ ]:
# Datos sintéticos: 3 clusters bien separados
rng = np.random.default_rng(0)
X_demo = np.vstack([
    rng.normal(loc=[0, 0],  scale=0.5, size=(50, 2)),
    rng.normal(loc=[5, 5],  scale=0.5, size=(50, 2)),
    rng.normal(loc=[0, 5],  scale=0.5, size=(50, 2)),
])

Z0_demo = X_demo[rng.choice(150, 3, replace=False)]  # 3 puntos aleatorios
Z_demo, labels_demo, J_demo = k_means_fit(X_demo, Z0_demo, NITERMAX=100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Clusters
ax = axes[0]
for j in range(3):
    mask = labels_demo == j
    ax.scatter(X_demo[mask, 0], X_demo[mask, 1], s=20, alpha=0.6, label=f"Cluster {j}")
ax.scatter(Z_demo[:, 0], Z_demo[:, 1], c='k', marker='X', s=150, zorder=5, label='Centroides')
ax.set_title("Resultado del clustering")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.legend(fontsize=9)

# Convergencia
ax = axes[1]
ax.plot(range(1, len(J_demo) + 1), J_demo, 'o-', markersize=5, color='#2c7bb6')
ax.set_title("Convergencia de $J_{\\mathrm{clust}}$")
ax.set_xlabel("Iteración"); ax.set_ylabel("$J_{\\mathrm{clust}}$")

plt.tight_layout()
plt.show()


> **2 b)** Use la base de datos `datakmeans.csv` para testear su algoritmo con $k=5$ representantes.
>
> **2 c)** Use la base de datos de imágenes de dígitos de MNIST con $k=20$ representantes.


### 2 b) y 2 c) — *Pendiente*

Estos incisos requieren los archivos `datakmeans.csv` y datos MNIST, respectivamente.  
Se implementarán una vez que los datos estén disponibles.


> **2 d)** Discuta por qué el algoritmo de K-means converge, es decir, ¿es seguro que de una
> iteración a la siguiente la función objetivo decrece?


### 2 d) ¿Por qué converge K-means?

El algoritmo minimiza la función objetivo

$$
J_{\text{clust}} = \sum_{j=1}^{k} \sum_{i \in C_j} \|\mathbf{x}_i - \mathbf{z}_j\|^2.
$$

En cada iteración se ejecutan dos pasos, y **cada uno garantiza que $J_{\text{clust}}$ no aumente**:

1. **Paso de asignación.** Se reasigna cada punto $\mathbf{x}_i$ al centroide más cercano. Como se elige el representante que minimiza $\|\mathbf{x}_i - \mathbf{z}_j\|^2$, el costo individual de cada punto no puede crecer, por lo que $J_{\text{clust}}$ no aumenta.

2. **Paso de actualización.** Se recalcula cada centroide como la media aritmética de los puntos de su cluster:
   $$\mathbf{z}_j^{\text{new}} = \frac{1}{|C_j|}\sum_{i \in C_j} \mathbf{x}_i.$$
   La media es, por definición, el minimizador de la suma de distancias al cuadrado dentro de cada grupo. Así, el costo por cluster tampoco crece.

Dado que:
- $J_{\text{clust}} \geq 0$ (acotada inferiormente),
- $J_{\text{clust}}$ es monótonamente no creciente,
- el número de particiones distintas de $N$ datos en $k$ clusters es **finito** ($\leq k^N$),

el algoritmo necesariamente **termina en un número finito de iteraciones**.

> **Observación.** K-means converge a un *mínimo local* de $J_{\text{clust}}$; el resultado puede depender de la inicialización $Z_0$. Por ello, en la práctica se recomienda ejecutar el algoritmo varias veces con distintas semillas y quedarse con la solución de menor costo.


---
## Problema 3 · Independencia lineal  
**(5 puntos)**


> **Enunciado.** Considere una matriz $A \in \mathbb{R}^{m \times n}$ con $m > n$ y un vector $\mathbf{b} \in \mathbb{R}^m$.
> Describa cómo puede asegurar de forma práctica si la solución del sistema $A\mathbf{x} = \mathbf{b}$ existe o no.
>
> Para $n=3$, $m=4$, aplique su método al sistema:
>
> $$x + y + z = 1, \quad 2x - 3y + 4z = 1, \quad 3x + 5y - z = 1, \quad 2y - 4z = -2.$$


### Método práctico — Teorema de Rouché–Frobenius

Para un sistema sobredeterminado $A\mathbf{x} = \mathbf{b}$ con $A \in \mathbb{R}^{m \times n}$, $m > n$, la solución existe si y solo si $\mathbf{b} \in \mathrm{Col}(A)$.

**Procedimiento.** Se construye la *matriz aumentada* $[A \mid \mathbf{b}]$ y se comparan rangos:

$$
\text{rango}(A) = \text{rango}\bigl([A \mid \mathbf{b}]\bigr)
\quad \Longleftrightarrow \quad
\text{el sistema es compatible (tiene solución).}
$$

Además, si $\text{rango}(A) = n$, la solución (de existir) es **única**.

En la práctica, el rango se calcula eficientemente mediante la **descomposición en valores singulares (SVD)**: el rango numérico es el número de valores singulares por encima de una tolerancia.


### Aplicación al sistema dado


In [ ]:
A = np.array([
    [1,  1,  1],
    [2, -3,  4],
    [3,  5, -1],
    [0,  2, -4]
], dtype=float)

b = np.array([1.0, 1.0, 1.0, -2.0])

# Matriz aumentada [A | b]
Ab = np.column_stack([A, b])

rank_A  = matrix_rank(A)
rank_Ab = matrix_rank(Ab)
n = A.shape[1]

print("Matriz A:")
print(A)
print(f"\nVector b = {b}")
print(f"\nrango(A)      = {rank_A}")
print(f"rango([A|b])  = {rank_Ab}")
print(f"n (incógnitas) = {n}")

if rank_A == rank_Ab:
    print(f"\n→ rango(A) = rango([A|b]) = {rank_A}  ⟹  el sistema ES compatible.")
    if rank_A == n:
        print(f"  Además rango(A) = n = {n}, por lo que la solución es ÚNICA.\n")
        x_sol = np.linalg.lstsq(A, b, rcond=None)[0]
        print(f"  Solución:  x = {x_sol[0]:.4f},  y = {x_sol[1]:.4f},  z = {x_sol[2]:.4f}")
        print(f"  Residuo ‖Ax − b‖ = {norm(A @ x_sol - b):.2e}")
else:
    print(f"\n→ rango(A) ≠ rango([A|b])  ⟹  el sistema NO tiene solución.")


---
## Problema 4 · Clasificador binario: Perceptrón  
**(15 puntos)**


> **4 a)** Programe el algoritmo del Perceptrón. Construya una rutina  
> `Perceptron_fit(X, y, nitmax, eta)` que tome los datos `X`, sus etiquetas `y`,
> y retorne los pesos `w` del clasificador.


### 4 a) Implementación de `Perceptron_fit`


In [ ]:
def Perceptron_fit(X, y, nitmax, eta):
    """
    Algoritmo del Perceptrón.

    Parámetros
    ----------
    X      : ndarray (N, d)    –  Datos de entrada.
    y      : ndarray (N,)     –  Etiquetas en {0, 1}.
    nitmax : int              –  Número máximo de épocas.
    eta    : float            –  Learning rate.

    Retorna
    -------
    w : ndarray (d+1,) – Pesos del hiperplano separador (incluye bias w₀).
    """
    N, d = X.shape
    y_signed = 2 * y - 1                          # {0,1} → {−1,+1}
    X_aug = np.column_stack([np.ones(N), X])      # columna de 1s para el bias
    w = np.zeros(d + 1)

    for epoch in range(nitmax):
        errors = 0
        for i in range(N):
            if y_signed[i] * (X_aug[i] @ w) <= 0:
                w += eta * y_signed[i] * X_aug[i]
                errors += 1
        if errors == 0:
            break

    return w


> **4 b)** Programe una rutina que, dado un vector de pesos `w` y un conjunto de datos `X`,
> prediga a qué clase pertenece cada dato.


### 4 b) Función de predicción


In [ ]:
def Perceptron_predict(X, w):
    """
    Predice la clase {0, 1} de cada dato.

    Parámetros
    ----------
    X : ndarray (N, d)   –  Datos.
    w : ndarray (d+1,)   –  Pesos (incluye bias).

    Retorna
    -------
    y_pred : ndarray (N,) – Etiquetas predichas en {0, 1}.
    """
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    return ((np.sign(X_aug @ w) + 1) / 2).astype(int)


> **4 c)** Programe una rutina que, dado `w`, `X` y sus etiquetas `y`, entregue un *score*
> de qué tan bien clasificados están los datos.


### 4 c) Función de *score*


In [ ]:
def Perceptron_score(X, y, w):
    """
    Accuracy del clasificador: proporción de datos correctamente clasificados.

    Retorna
    -------
    score : float ∈ [0, 1]
    """
    return np.mean(Perceptron_predict(X, w) == y)


**Verificación** con la compuerta lógica AND (datos linealmente separables):


In [ ]:
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

w_and = Perceptron_fit(X_and, y_and, nitmax=100, eta=0.1)

print("  Datos       Etiqueta real   Predicción")
print("  ──────────  ─────────────   ──────────")
for xi, yi, yp in zip(X_and, y_and, Perceptron_predict(X_and, w_and)):
    print(f"  {xi}          {yi}               {yp}")
print(f"\n  Score (accuracy) = {Perceptron_score(X_and, y_and, w_and):.0%}")


> **4 d)** y **4 e)** requieren los archivos `datos1.csv` y `datos2.csv`.  
> Se implementarán una vez que los datos estén disponibles.


---
## Problema 5 · Descomposición en valores singulares  
**(5 puntos)**


> **Enunciado.** Muestre que para $A \in \mathbb{R}^{m \times n}$ con valores singulares
> $\sigma_1, \dots, \sigma_p$, $p = \min\{m, n\}$, se cumple
>
> $$\|A\|_F = \sqrt{\sigma_1^2 + \cdots + \sigma_p^2}.$$
>
> Verifique para la matriz  
> $A = \begin{pmatrix} 4 & 1 & 0 \\ 1 & 4 & 1 \\ 0 & 1 & 4 \end{pmatrix}$.


### Demostración

Sea $A = U\Sigma V^{\!\top}$ la SVD de $A$, con $U$, $V$ matrices ortogonales y $\Sigma$ la matriz diagonal de valores singulares.

Usamos que $\|A\|_F^2 = \mathrm{tr}(A^{\!\top}\!A)$ y que la traza es invariante cíclica:

$$
\|A\|_F^2
= \mathrm{tr}(A^{\!\top}\!A)
= \mathrm{tr}\!\bigl(V\Sigma^{\!\top}U^{\!\top}U\Sigma V^{\!\top}\bigr)
= \mathrm{tr}\!\bigl(V\Sigma^{\!\top}\Sigma V^{\!\top}\bigr)
= \mathrm{tr}\!\bigl(\Sigma^{\!\top}\Sigma V^{\!\top}V\bigr)
= \mathrm{tr}\!\bigl(\Sigma^{\!\top}\Sigma\bigr).
$$

La matriz $\Sigma^{\!\top}\Sigma \in \mathbb{R}^{n\times n}$ es diagonal con entradas $\sigma_1^2, \dots, \sigma_p^2$ (y ceros si $n > m$). Luego:

$$
\|A\|_F^2 = \sigma_1^2 + \sigma_2^2 + \cdots + \sigma_p^2
\qquad \Longrightarrow \qquad
\boxed{\;\|A\|_F = \sqrt{\sigma_1^2 + \cdots + \sigma_p^2}\;}
\qquad \blacksquare
$$


### Verificación numérica


In [ ]:
A = np.array([
    [4, 1, 0],
    [1, 4, 1],
    [0, 1, 4]
], dtype=float)

U, sigma, Vt = svd(A)

frob_direct = norm(A, 'fro')
frob_svd    = np.sqrt(np.sum(sigma ** 2))

print("Matriz A:")
print(A)
print(f"\nValores singulares:  σ = {sigma}")
print(f"\n  ‖A‖_F  (directa)   = {frob_direct:.10f}")
print(f"  √(Σ σᵢ²) (vía SVD) = {frob_svd:.10f}")
print(f"  |diferencia|        = {abs(frob_direct - frob_svd):.2e}")
print(f"\n  ✓ Identidad verificada." if np.isclose(frob_direct, frob_svd) else "  ✗ ERROR")


---
## Problema 6 · Análisis de Componentes Principales (PCA)  
**(15 puntos)**

> Todos los incisos de este problema requieren la base de datos MNIST.  
> Se implementarán una vez que los datos estén disponibles.
